# Value-Based Healthcare Decision Intelligence

## Resource allocation policy evaluation with CMS-calibrated simulation

This workflow evaluates how a fixed resource envelope can be distributed among preventive care, chronic disease management, care coordination, and readmission prevention. It combines public hospital benchmarks, a configurable simulation environment, and reinforcement learning to compare allocation policies.

**Decision outputs:** policy comparisons, spending and readmission trade-offs, efficiency trajectories, learning diagnostics, and hyperparameter selection across multiple seeds.

**Operating boundary:** CMS observations establish baseline conditions and variability. Allocation effects are explicit simulation assumptions; results support scenario exploration and require organization-specific validation before operational use.

The implementation uses 81 discrete states, six allocation actions, Q-Learning and Dyna-Q planning, and a two-stage search across 108 hyperparameter configurations.


## 1. Reinforcement Learning Formulation

The problem is modeled as a Markov Decision Process:

\[
\mathcal{M}=(S,A,P,R,\gamma)
\]

The state is:

\[
S_t=(C_t,D_t,Q_t,RR_t)
\]

where:
- \(C_t\) = cost/efficiency condition derived from MSPB
- \(D_t\) = demand/volume condition derived from discharge volume
- \(Q_t\) = value/efficiency condition derived from HVBP efficiency
- \(RR_t\) = readmission condition derived from the excess readmission ratio

Each variable has three empirical categories: Low, Medium, and High.

Therefore:

\[
|S|=3^4=81
\]

Dyna-Q uses the Q-Learning update:

\[
Q(S_t,A_t)\leftarrow Q(S_t,A_t)+
\alpha\left[
R_{t+1}+\gamma\max_{a'}Q(S_{t+1},a')-Q(S_t,A_t)
\right]
\]

and then performs additional planning updates from its learned model.

In [ ]:

import io
import math
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
RNG = np.random.default_rng(SEED)

# CMS Provider Data Catalog short IDs
CMS_DATASETS = {
    "hrrp": {
        "id": "9n3s-kdb3",
        "title": "Hospital Readmissions Reduction Program",
    },
    "mspb": {
        "id": "rrqw-56er",
        "title": "Medicare Spending Per Beneficiary - Hospital",
    },
    "hvbp_eff": {
        "id": "su9h-3pvj",
        "title": "Hospital Value-Based Purchasing (HVBP) - Efficiency Scores",
    },
}

PROGRAMS = [
    "Preventive Care",
    "Chronic Disease Management",
    "Care Coordination",
    "Readmission Prevention",
]

ACTIONS = np.array([
    [0.25, 0.25, 0.25, 0.25],
    [0.30, 0.30, 0.20, 0.20],
    [0.20, 0.35, 0.25, 0.20],
    [0.20, 0.25, 0.35, 0.20],
    [0.20, 0.25, 0.20, 0.35],
    [0.30, 0.20, 0.20, 0.30],
], dtype=float)

ACTION_NAMES = [
    "Equal Allocation",
    "Preventive + Chronic Focus",
    "Chronic Disease Focus",
    "Care Coordination Focus",
    "Readmission Prevention Focus",
    "Preventive + Readmission Focus",
]

STATE_DIMS = (3, 3, 3, 3)
N_STATES = int(np.prod(STATE_DIMS))
N_ACTIONS = len(ACTIONS)

print("States:", N_STATES)
print("Actions:", N_ACTIONS)



## 2. CMS Real-Data Loader

The loader first attempts to read the official CMS Provider Data API directly.

If the API is blocked by a restricted network, download the three CSV files from CMS and place them beside the notebook. The same pipeline can read those local files instead.


In [ ]:

def _normalize_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")
        for c in df.columns
    ]
    return df


def fetch_cms_provider_dataset(short_id, page_size=5000, max_pages=1000):
    '''
    Fetch a CMS Provider Data Catalog dataset using the provider-data API.
    Returns a pandas DataFrame.
    '''
    base = (
        "https://data.cms.gov/provider-data/api/1/datastore/query/"
        f"{short_id}/0"
    )

    rows = []
    offset = 0
    seen_pages = set()

    for _ in range(max_pages):
        params = {
            "offset": offset,
            "count": "true",
            "results": "true",
            "schema": "true",
            "keys": "true",
            "format": "json",
        }
        response = requests.get(base, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()

        # CMS endpoint responses have changed shape over time.
        batch = None
        if isinstance(payload, dict):
            for key in ["results", "data", "rows"]:
                if key in payload and isinstance(payload[key], list):
                    batch = payload[key]
                    break
            if batch is None and "response" in payload:
                nested = payload["response"]
                if isinstance(nested, dict):
                    for key in ["results", "data", "rows"]:
                        if key in nested and isinstance(nested[key], list):
                            batch = nested[key]
                            break

        if not batch:
            break

        signature = repr(batch)
        if signature in seen_pages:
            raise RuntimeError("CMS pagination repeated a page; use archived CSV exports.")
        seen_pages.add(signature)
        rows.extend(batch)

        offset += len(batch)
    else:
        raise RuntimeError("CMS pagination limit reached; refusing to use a partial dataset.")

    if not rows:
        raise RuntimeError(
            f"No rows returned for CMS dataset {short_id}. "
            "Use the local-CSV fallback cell below."
        )

    return _normalize_columns(pd.DataFrame(rows))


def load_local_csv(path):
    return _normalize_columns(pd.read_csv(path, dtype=str))


def load_cms_data(local_paths=None):
    '''
    local_paths example:
    {
        "hrrp": "Hospital_Readmissions_Reduction_Program.csv",
        "mspb": "Medicare_Spending_Per_Beneficiary_Hospital.csv",
        "hvbp_eff": "HVBP_Efficiency_Scores.csv",
    }
    '''
    result = {}

    for key, meta in CMS_DATASETS.items():
        if local_paths and key in local_paths and Path(local_paths[key]).exists():
            print(f"Loading local file for {key}: {local_paths[key]}")
            result[key] = load_local_csv(local_paths[key])
        else:
            print(f"Fetching {meta['title']} from CMS...")
            result[key] = fetch_cms_provider_dataset(meta["id"])

        print(key, result[key].shape)

    return result



### Run the real-data load

If direct API access works, leave `LOCAL_PATHS = None`.

If you downloaded the three CMS CSV files manually, replace the filenames below and uncomment the dictionary.


In [ ]:
import json

import os
LOCAL_PATHS = json.loads(os.environ["VBHC_LOCAL_PATHS"]) if os.environ.get("VBHC_LOCAL_PATHS") else None


# Example fallback:
# LOCAL_PATHS = {
#     "hrrp": "Hospital_Readmissions_Reduction_Program.csv",
#     "mspb": "Medicare_Spending_Per_Beneficiary_Hospital.csv",
#     "hvbp_eff": "HVBP_Efficiency_Scores.csv",
# }

cms = load_cms_data(LOCAL_PATHS)

# Retain the exact input tables for subsequent local replay.
SNAPSHOT = Path("cms_snapshot")
SNAPSHOT.mkdir(exist_ok=True)
for source_name, source_table in cms.items():
    source_table.to_csv(SNAPSHOT / f"{source_name}.csv", index=False)
print("CMS input snapshot:", SNAPSHOT.resolve())

hrrp_raw = cms["hrrp"]
mspb_raw = cms["mspb"]
hvbp_raw = cms["hvbp_eff"]

print("\nHRRP columns:")
print(hrrp_raw.columns.tolist())
print("\nMSPB columns:")
print(mspb_raw.columns.tolist())
print("\nHVBP Efficiency columns:")
print(hvbp_raw.columns.tolist())



## 3. Robust Column Detection

CMS column labels may change slightly between releases.  
The helper functions below search for likely fields instead of relying on one exact column spelling.


In [ ]:
def find_col(df, candidates, required=True):
    cols = list(df.columns)

    # Exact normalized matches first
    for candidate in candidates:
        candidate = candidate.lower()
        if candidate in cols:
            return candidate

    # Contains-all-token matches
    for candidate in candidates:
        tokens = candidate.lower().split()
        for col in cols:
            if all(token in col for token in tokens):
                return col

    if required:
        raise KeyError(
            f"Could not find any of {candidates}. Available columns: {cols}"
        )
    return None


def numeric(series):
    series_str = series.astype(str)

    # Handle 'X out of Y' format
    def parse_fraction(val):
        match = re.match(r"(\d+)\s+out of\s+(\d+)", val)
        if match:
            numerator = float(match.group(1))
            denominator = float(match.group(2))
            if denominator != 0:
                return numerator / denominator
            else:
                return np.nan # Division by zero
        return val # Return original if no match

    processed_series = series_str.apply(parse_fraction)

    return pd.to_numeric(
        processed_series.astype(str).str.replace(",", "", regex=False)
                          .str.replace("%", "", regex=False)
                          .replace({"Not Available": np.nan,
                                    "Not Applicable": np.nan,
                                    "nan": np.nan}),
        errors="coerce"
    )


## 4. Prepare the CMS Hospital Measures

### HRRP
HRRP typically contains multiple rows per hospital, one for each readmission measure.  
We aggregate the excess readmission ratio across available conditions. A value above 1 indicates excess readmissions relative to expected performance.

### MSPB
The MSPB score reflects Medicare spending for an episode of care relative to the national benchmark.

### HVBP Efficiency
The efficiency score provides a value-based purchasing efficiency/cost-reduction measure.


In [ ]:

# -------- HRRP --------
hrrp_facility = find_col(
    hrrp_raw,
    ["facility_id", "facility id", "provider_id", "provider id"]
)
hrrp_name = find_col(
    hrrp_raw,
    ["facility_name", "facility name", "hospital_name", "hospital name"],
    required=False
)
hrrp_state = find_col(hrrp_raw, ["state"], required=False)
hrrp_ratio = find_col(
    hrrp_raw,
    ["excess_readmission_ratio", "excess readmission ratio"]
)
hrrp_discharges = find_col(
    hrrp_raw,
    ["number_of_discharges", "number of discharges", "discharges"],
    required=False
)

hrrp = pd.DataFrame({
    "facility_id": hrrp_raw[hrrp_facility].astype(str).str.zfill(6),
    "excess_readmission_ratio": numeric(hrrp_raw[hrrp_ratio]),
})

if hrrp_name:
    hrrp["facility_name"] = hrrp_raw[hrrp_name]
if hrrp_state:
    hrrp["state"] = hrrp_raw[hrrp_state]
if hrrp_discharges:
    hrrp["discharges"] = numeric(hrrp_raw[hrrp_discharges])

agg_map = {
    "excess_readmission_ratio": "mean",
}
if "facility_name" in hrrp:
    agg_map["facility_name"] = "first"
if "state" in hrrp:
    agg_map["state"] = "first"
if "discharges" in hrrp:
    agg_map["discharges"] = "sum"

hrrp_hospital = (
    hrrp.dropna(subset=["excess_readmission_ratio"])
        .groupby("facility_id", as_index=False)
        .agg(agg_map)
)

# -------- MSPB --------
mspb_facility = find_col(
    mspb_raw,
    ["facility_id", "facility id", "provider_id", "provider id"]
)
mspb_score_col = find_col(
    mspb_raw,
    ["score", "mspb_score", "mspb score"]
)

mspb = pd.DataFrame({
    "facility_id": mspb_raw[mspb_facility].astype(str).str.zfill(6),
    "mspb_score": numeric(mspb_raw[mspb_score_col]),
})

# Some CMS datasets have more than one row per facility.
mspb_hospital = (
    mspb.dropna(subset=["mspb_score"])
        .groupby("facility_id", as_index=False)["mspb_score"]
        .mean()
)

# -------- HVBP efficiency --------
hvbp_facility = find_col(
    hvbp_raw,
    ["facility_id", "facility id", "provider_id", "provider id"]
)

# Try several likely score fields, preferring a domain/efficiency score.
hvbp_score_col = find_col(
    hvbp_raw,
    [
        "mspb1_measure_score",
        "efficiency_and_cost_reduction_domain_score",
        "efficiency cost reduction domain score",
        "efficiency_score",
        "efficiency score",
        "score",
    ]
)

hvbp = pd.DataFrame({
    "facility_id": hvbp_raw[hvbp_facility].astype(str).str.zfill(6),
    "hvbp_efficiency_score": numeric(hvbp_raw[hvbp_score_col]),
})

hvbp_hospital = (
    hvbp.dropna(subset=["hvbp_efficiency_score"])
        .groupby("facility_id", as_index=False)["hvbp_efficiency_score"]
        .mean()
)

print("HRRP hospitals:", len(hrrp_hospital))
print("MSPB hospitals:", len(mspb_hospital))
print("HVBP hospitals:", len(hvbp_hospital))
for name, table in [("HRRP", hrrp_hospital), ("MSPB", mspb_hospital), ("HVBP", hvbp_hospital)]:
    if table.empty:
        raise ValueError(f"{name} has no usable hospital measures. Inspect the source columns and score values before training.")



## 5. Merge the Real CMS Data

The CMS Certification Number / Facility ID is used to join datasets.


In [ ]:

real_hospitals = (
    hrrp_hospital
    .merge(mspb_hospital, on="facility_id", how="inner")
    .merge(hvbp_hospital, on="facility_id", how="inner")
)

# Clean impossible/nonreported values
for col in [
    "excess_readmission_ratio",
    "mspb_score",
    "hvbp_efficiency_score",
]:
    real_hospitals = real_hospitals[
        np.isfinite(real_hospitals[col])
    ]

real_hospitals = real_hospitals.reset_index(drop=True)

if real_hospitals.empty:
    raise ValueError("No hospitals remain after joining CMS measures. Check facility IDs, score parsing, and release compatibility.")

print("Merged hospitals:", real_hospitals.shape)
display(real_hospitals.head())
display(real_hospitals.describe(include="all"))



## 6. Derive RL State Categories from the Real Data

Rather than inventing Low/Medium/High thresholds, this notebook uses the CMS hospital distribution.

For each measure:
- Low = below the 33rd percentile
- Medium = 33rd to 67th percentile
- High = above the 67th percentile

For unfavorable measures such as MSPB spending and excess readmissions, a **high numeric value represents a worse condition**.


In [ ]:

def tertile_cutpoints(series):
    clean = series.dropna()
    return tuple(clean.quantile([1/3, 2/3]).values)

READMIT_CUTS = tertile_cutpoints(real_hospitals["excess_readmission_ratio"])
MSPB_CUTS = tertile_cutpoints(real_hospitals["mspb_score"])
HVBP_CUTS = tertile_cutpoints(real_hospitals["hvbp_efficiency_score"])

if "discharges" in real_hospitals.columns:
    DISCHARGE_CUTS = tertile_cutpoints(real_hospitals["discharges"])
else:
    DISCHARGE_CUTS = (0.9, 1.1)  # fallback proxy only if CMS release omits discharge count

print("Readmission cutpoints:", READMIT_CUTS)
print("MSPB cutpoints:", MSPB_CUTS)
print("HVBP efficiency cutpoints:", HVBP_CUTS)
print("Discharge cutpoints:", DISCHARGE_CUTS)


## 7. Convert Each Real Hospital to an Initial RL State

The model uses four state dimensions:

1. **Cost condition** → MSPB score
2. **Demand/volume condition** → discharge volume
3. **Value/efficiency condition** → HVBP efficiency score
4. **Readmission condition** → excess readmission ratio

No derived pressure proxy is added; MSPB and readmission performance are already represented directly.

Using four three-level dimensions gives:

\[
3^4=81
\]

possible tabular states.

In [ ]:
def category(value, cuts):
    if value < cuts[0]:
        return 0
    if value < cuts[1]:
        return 1
    return 2


def hospital_to_state(row):
    cost_state = category(row["mspb_score"], MSPB_CUTS)

    if "discharges" in row.index and pd.notna(row["discharges"]):
        demand_state = category(row["discharges"], DISCHARGE_CUTS)
    else:
        demand_state = 1

    value_state = category(row["hvbp_efficiency_score"], HVBP_CUTS)
    readmit_state = category(
        row["excess_readmission_ratio"],
        READMIT_CUTS
    )

    return (
        cost_state,
        demand_state,
        value_state,
        readmit_state,
    )


real_hospitals["initial_state"] = real_hospitals.apply(
    hospital_to_state,
    axis=1
)

real_hospitals["state_index"] = real_hospitals["initial_state"].apply(
    lambda s: int(np.ravel_multi_index(s, STATE_DIMS))
)

print("Reachable empirical states:", real_hospitals["state_index"].nunique())
print("Theoretical states:", N_STATES)
print(
    "Empirical state coverage:",
    f'{real_hospitals["state_index"].nunique()/N_STATES:.1%}'
)

display(
    real_hospitals[
        [
            "facility_id",
            "mspb_score",
            "hvbp_efficiency_score",
            "excess_readmission_ratio",
            "discharges" if "discharges" in real_hospitals.columns else "facility_id",
            "initial_state",
            "state_index",
        ]
    ].head(10)
)

## 8. CMS-Calibrated Environment

At reset, a real hospital is sampled from the CMS dataset. Its MSPB, HVBP efficiency, readmission performance, and discharge volume define the starting state.

### Transition assumptions
Assumed action effects allow simulated allocation decisions to move hospital measures across empirical Low/Medium/High state thresholds. The effects remain small relative to the CMS cross-hospital distributions.

### Reward definition

The reward contains three performance measures, with conceptual overlap between MSPB and HVBP efficiency:

\[
R_{t+1}
=
0.35(\Delta MSPB_t)
+
0.35(\Delta RR_t)
+
0.30(\Delta HVBP_t)
\]

where:
- lower MSPB is better,
- lower excess readmission ratio is better,
- higher HVBP efficiency is better.

Each change is standardized using the observed CMS population variability and clipped to \([-1,1]\).

In [ ]:
class CMSCalibratedVBHCEnv:
    def __init__(self, hospital_data, seed=42, periods=12):
        self.data = hospital_data.reset_index(drop=True).copy()
        self.rng = np.random.default_rng(seed)
        self.periods = periods

        # Empirical variability from the CMS hospital distribution.
        self.readmit_sd = max(
            self.data["excess_readmission_ratio"].std(ddof=0),
            0.01
        )
        self.mspb_sd = max(
            self.data["mspb_score"].std(ddof=0),
            0.01
        )
        self.hvbp_sd = max(
            self.data["hvbp_efficiency_score"].std(ddof=0),
            0.05
        )

        self.mspb_median = self.data["mspb_score"].median()
        self.readmit_median = self.data["excess_readmission_ratio"].median()
        self.hvbp_median = self.data["hvbp_efficiency_score"].median()

        self.mspb_bounds = (
            self.data["mspb_score"].quantile(0.01),
            self.data["mspb_score"].quantile(0.99),
        )
        self.readmit_bounds = (
            self.data["excess_readmission_ratio"].quantile(0.01),
            self.data["excess_readmission_ratio"].quantile(0.99),
        )
        self.hvbp_bounds = (
            self.data["hvbp_efficiency_score"].quantile(0.01),
            self.data["hvbp_efficiency_score"].quantile(0.99),
        )

    def reset(self, seed=None, hospital_index=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        if hospital_index is None:
            hospital_index = int(self.rng.integers(len(self.data)))

        row = self.data.iloc[hospital_index]
        self.facility_id = row["facility_id"]
        self.period = 0

        self.mspb = float(row["mspb_score"])
        self.readmission = float(row["excess_readmission_ratio"])
        self.hvbp = float(row["hvbp_efficiency_score"])

        if "discharges" in row.index and pd.notna(row["discharges"]):
            self.volume = float(row["discharges"])
        else:
            self.volume = 1.0

        return self._state()

    def _state(self):
        return (
            category(self.mspb, MSPB_CUTS),
            (
                category(self.volume, DISCHARGE_CUTS)
                if "discharges" in self.data.columns
                else 1
            ),
            category(self.hvbp, HVBP_CUTS),
            category(self.readmission, READMIT_CUTS),
        )

    def step(self, action_idx):
        shares = ACTIONS[action_idx]
        delta = shares - 0.25

        old_mspb = self.mspb
        old_readmission = self.readmission
        old_hvbp = self.hvbp

        # Noise remains tied to real CMS variability.
        mspb_noise = self.rng.normal(0, 0.045 * self.mspb_sd)
        readmit_noise = self.rng.normal(0, 0.045 * self.readmit_sd)
        hvbp_noise = self.rng.normal(0, 0.045 * self.hvbp_sd)

        # Assumed allocation-response coefficients for scenario exploration.
        # Directions reflect the modeled strategic mechanism; they are
        # simulation assumptions, not causal estimates from CMS.
        mspb_effect = (
            -0.020 * delta[0]
            -0.028 * delta[1]
            -0.024 * delta[2]
            -0.012 * delta[3]
        )

        readmit_effect = (
            -0.008 * delta[1]
            -0.014 * delta[2]
            -0.028 * delta[3]
        )

        hvbp_effect = (
             0.70 * delta[0]
            +0.90 * delta[1]
            +1.10 * delta[2]
            +0.90 * delta[3]
        )

        # Mild pull toward the empirical CMS population center prevents drift.
        self.mspb += (
            0.05 * (self.mspb_median - self.mspb)
            + mspb_effect
            + mspb_noise
        )

        self.readmission += (
            0.05 * (self.readmit_median - self.readmission)
            + readmit_effect
            + readmit_noise
        )

        self.hvbp += (
            0.05 * (self.hvbp_median - self.hvbp)
            + hvbp_effect
            + hvbp_noise
        )

        self.mspb = float(np.clip(
            self.mspb,
            self.mspb_bounds[0],
            self.mspb_bounds[1],
        ))
        self.readmission = float(np.clip(
            self.readmission,
            self.readmit_bounds[0],
            self.readmit_bounds[1],
        ))
        self.hvbp = float(np.clip(
            self.hvbp,
            self.hvbp_bounds[0],
            self.hvbp_bounds[1],
        ))

        # Standardized improvement components.
        # Lower MSPB/readmission are better; higher HVBP is better.
        mspb_improvement = (
            old_mspb - self.mspb
        ) / max(self.mspb_sd, 1e-9)

        readmit_improvement = (
            old_readmission - self.readmission
        ) / max(self.readmit_sd, 1e-9)

        hvbp_improvement = (
            self.hvbp - old_hvbp
        ) / max(self.hvbp_sd, 1e-9)

        reward = float(
            0.35 * np.clip(mspb_improvement, -1, 1)
            + 0.35 * np.clip(readmit_improvement, -1, 1)
            + 0.30 * np.clip(hvbp_improvement, -1, 1)
        )

        self.period += 1
        done = self.period >= self.periods

        metrics = {
            "facility_id": self.facility_id,
            "period": self.period,
            "action": action_idx,
            "action_name": ACTION_NAMES[action_idx],
            "mspb_score": self.mspb,
            "excess_readmission_ratio": self.readmission,
            "hvbp_efficiency_score": self.hvbp,
            "reward": reward,
        }

        return self._state(), reward, done, metrics

## 9. Dyna-Q Agent

The agent combines:

- epsilon-greedy acting,
- direct Q-Learning,
- model learning,
- planning from stored transitions.

Setting the number of planning steps to **0** gives the direct Q-Learning comparison condition within the same implementation.

In [ ]:

def state_to_index(state):
    return int(np.ravel_multi_index(state, STATE_DIMS))


class DynaQAgent:
    def __init__(
        self,
        alpha=0.10,
        gamma=0.95,
        epsilon=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.995,
        planning_steps=10,
        seed=42,
    ):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.planning_steps = planning_steps
        self.rng = np.random.default_rng(seed)

        self.q = np.zeros((N_STATES, N_ACTIONS), dtype=float)
        self.model = {}

    def select_action(self, state_index, explore=True):
        if explore and self.rng.random() < self.epsilon:
            return int(self.rng.integers(N_ACTIONS)), True

        values = self.q[state_index]
        best = np.flatnonzero(np.isclose(values, values.max()))
        return int(self.rng.choice(best)), False

    def update(self, s, a, r, s_next, done):
        target = r if done else r + self.gamma * self.q[s_next].max()
        td_error = target - self.q[s, a]
        self.q[s, a] += self.alpha * td_error
        return float(td_error)

    def remember(self, s, a, r, s_next, done):
        key = (s, a)
        self.model.setdefault(key, []).append(
            (float(r), int(s_next), bool(done))
        )
        if len(self.model[key]) > 50:
            self.model[key].pop(0)

    def plan(self):
        if self.planning_steps == 0 or not self.model:
            return []

        keys = list(self.model.keys())
        errors = []

        for _ in range(self.planning_steps):
            s, a = keys[int(self.rng.integers(len(keys)))]
            outcomes = self.model[(s, a)]
            r, s_next, done = outcomes[
                int(self.rng.integers(len(outcomes)))
            ]
            errors.append(self.update(s, a, r, s_next, done))

        return errors

    def decay(self):
        self.epsilon = max(
            self.epsilon_min,
            self.epsilon * self.epsilon_decay
        )


## 10. Primary Dyna-Q Training Run

In [ ]:
def train_dynaq(
    hospital_data,
    episodes=3000,
    planning_steps=10,
    seed=42,
    alpha=0.10,
    gamma=0.95,
    epsilon_decay=0.995,
    epsilon_min=0.05,
    keep_step_records=True,
):
    env = CMSCalibratedVBHCEnv(hospital_data, seed=seed, periods=12)

    agent = DynaQAgent(
        alpha=alpha,
        gamma=gamma,
        epsilon=1.0,
        epsilon_min=epsilon_min,
        epsilon_decay=epsilon_decay,
        planning_steps=planning_steps,
        seed=seed,
    )

    episode_records = []
    step_records = []

    for episode in range(1, episodes + 1):
        state = env.reset(seed=seed + episode)
        s = state_to_index(state)

        total_reward = 0.0
        discounted_return = 0.0
        direct_errors = []
        planning_errors = []
        exploratory_actions = 0

        for t in range(env.periods):
            action, exploratory = agent.select_action(s, explore=True)
            exploratory_actions += int(exploratory)

            next_state, reward, done, metrics = env.step(action)
            s_next = state_to_index(next_state)

            td = agent.update(s, action, reward, s_next, done)
            direct_errors.append(abs(td))

            agent.remember(s, action, reward, s_next, done)
            p_errors = agent.plan()
            planning_errors.extend(abs(e) for e in p_errors)

            total_reward += reward
            discounted_return += (agent.gamma ** t) * reward

            if keep_step_records:
                step_records.append({
                    "episode": episode,
                    "step": t + 1,
                    "state_index": s,
                    "next_state_index": s_next,
                    "epsilon": agent.epsilon,
                    "exploratory_action": int(exploratory),
                    "direct_td_error": td,
                    **metrics,
                })

            s = s_next

            if done:
                break

        agent.decay()

        episode_records.append({
            "episode": episode,
            "episode_reward": total_reward,
            "discounted_return": discounted_return,
            "avg_abs_direct_td_error": np.mean(direct_errors),
            "avg_abs_planning_td_error": (
                np.mean(planning_errors)
                if planning_errors
                else 0.0
            ),
            "exploration_rate": exploratory_actions / env.periods,
            "epsilon_end": agent.epsilon,
        })

    return (
        agent,
        pd.DataFrame(episode_records),
        pd.DataFrame(step_records),
    )


# Primary run remains the previously selected Dyna-Q configuration.
agent, training_episodes, training_steps = train_dynaq(
    real_hospitals,
    episodes=3000,
    planning_steps=10,
    seed=SEED,
    alpha=0.10,
    gamma=0.95,
    epsilon_decay=0.995,
)

display(training_episodes.tail())


## 11. Evaluation Against an Equal-Allocation Baseline

The trained policy is evaluated with exploration disabled.

The comparison baseline allocates 25% to each strategic area.


In [ ]:

def evaluate_agent(agent, hospital_data, episodes=500, seed=10_000):
    env = CMSCalibratedVBHCEnv(hospital_data, seed=seed, periods=12)
    records = []

    for episode in range(1, episodes + 1):
        state = env.reset(seed=seed + episode)
        s = state_to_index(state)

        total_reward = 0.0
        discounted_return = 0.0
        last_metrics = None

        for t in range(env.periods):
            action, _ = agent.select_action(s, explore=False)
            next_state, reward, done, metrics = env.step(action)
            s = state_to_index(next_state)

            total_reward += reward
            discounted_return += (agent.gamma ** t) * reward
            last_metrics = metrics

            if done:
                break

        records.append({
            "episode": episode,
            "episode_reward": total_reward,
            "discounted_return": discounted_return,
            "final_mspb_score": last_metrics["mspb_score"],
            "final_readmission_ratio": last_metrics["excess_readmission_ratio"],
            "final_hvbp_efficiency": last_metrics["hvbp_efficiency_score"],
        })

    return pd.DataFrame(records)


def evaluate_equal_baseline(hospital_data, episodes=500, seed=20_000, gamma=0.95):
    env = CMSCalibratedVBHCEnv(hospital_data, seed=seed, periods=12)
    records = []

    for episode in range(1, episodes + 1):
        env.reset(seed=seed + episode)

        total_reward = 0.0
        discounted_return = 0.0
        last_metrics = None

        for t in range(env.periods):
            _, reward, done, metrics = env.step(0)
            total_reward += reward
            discounted_return += (gamma ** t) * reward
            last_metrics = metrics
            if done:
                break

        records.append({
            "episode": episode,
            "episode_reward": total_reward,
            "discounted_return": discounted_return,
            "final_mspb_score": last_metrics["mspb_score"],
            "final_readmission_ratio": last_metrics["excess_readmission_ratio"],
            "final_hvbp_efficiency": last_metrics["hvbp_efficiency_score"],
        })

    return pd.DataFrame(records)


dynaq_eval = evaluate_agent(agent, real_hospitals)
baseline_eval = evaluate_equal_baseline(real_hospitals, seed=10_000, gamma=agent.gamma)

comparison = pd.DataFrame({
    "Metric": [
        "Average episode reward",
        "Average discounted return",
        "Final MSPB score",
        "Final excess readmission ratio",
        "Final normalized HVBP MSPB measure score",
    ],
    "Dyna-Q": [
        dynaq_eval["episode_reward"].mean(),
        dynaq_eval["discounted_return"].mean(),
        dynaq_eval["final_mspb_score"].mean(),
        dynaq_eval["final_readmission_ratio"].mean(),
        dynaq_eval["final_hvbp_efficiency"].mean(),
    ],
    "Equal Allocation": [
        baseline_eval["episode_reward"].mean(),
        baseline_eval["discounted_return"].mean(),
        baseline_eval["final_mspb_score"].mean(),
        baseline_eval["final_readmission_ratio"].mean(),
        baseline_eval["final_hvbp_efficiency"].mean(),
    ],
})

comparison["Difference"] = (
    comparison["Dyna-Q"] - comparison["Equal Allocation"]
)

display(comparison)



## 12. Policy Performance and Learning Diagnostics


In [ ]:

rolling_reward = training_episodes["episode_reward"].rolling(
    100, min_periods=20
).mean()

plt.figure(figsize=(10, 5))
plt.plot(
    training_episodes["episode"],
    training_episodes["episode_reward"],
    alpha=0.20,
    label="Episode reward",
)
plt.plot(
    training_episodes["episode"],
    rolling_reward,
    linewidth=2,
    label="100-episode rolling average",
)
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("CMS-Calibrated Dyna-Q Training Reward")
plt.legend()
plt.show()

rolling_td = training_episodes["avg_abs_direct_td_error"].rolling(
    100, min_periods=20
).mean()

plt.figure(figsize=(10, 5))
plt.plot(
    training_episodes["episode"],
    training_episodes["avg_abs_direct_td_error"],
    alpha=0.25,
)
plt.plot(
    training_episodes["episode"],
    rolling_td,
    linewidth=2,
)
plt.xlabel("Episode")
plt.ylabel("Average Absolute TD Error")
plt.title("Temporal-Difference Error")
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(
    training_episodes["episode"],
    training_episodes["epsilon_end"],
)
plt.xlabel("Episode")
plt.ylabel("Epsilon")
plt.title("Exploration Rate Decay")
plt.show()


## 13. Controlled Planning-Step and Sample-Efficiency Experiment

Planning configurations use common seed values:

- same training seed for every planning condition,
- same evaluation seed for every planning condition,
- same number of episodes,
- same hyperparameters except planning steps.

We compare:

\[
n \in \{0,1,5,10,20\}
\]

where \(n=0\) represents direct Q-Learning.

### Sample-efficiency threshold

First, the asymptotic reward of the \(n=0\) control is estimated using its final 100 training episodes.

A common threshold is then defined as:

\[
R_{threshold}=0.90\times R_{QL,\ final100}
\]

when the control asymptotic reward is positive.

For each planning condition, we report the first episode where the **100-episode rolling average** reaches that common threshold. Direct interactions are:

\[
\text{interactions}=\text{episode}\times12
\]

This compares interactions to an exploratory training-reward threshold. Because planning and action selection share a random stream, identical seeds do not guarantee identical exploration draws across planning counts. See the methodology for evaluation limits.

In [ ]:
def first_episode_to_threshold(
    rewards,
    threshold,
    window=100,
):
    rolling = rewards.rolling(
        window,
        min_periods=window
    ).mean()

    hits = np.where(rolling.values >= threshold)[0]

    if len(hits) == 0:
        return np.nan

    # episodes are 1-indexed
    return int(hits[0] + 1)


PLANNING_LEVELS = [0, 1, 5, 10, 20]
COMMON_TRAIN_SEED = 1234
COMMON_EVAL_SEED = 30_000
TRAIN_EPISODES = 2000
EVAL_EPISODES = 500

controlled_runs = {}

for n_plan in PLANNING_LEVELS:
    test_agent, test_train, test_steps = train_dynaq(
        real_hospitals,
        episodes=TRAIN_EPISODES,
        planning_steps=n_plan,
        seed=COMMON_TRAIN_SEED,
    )

    test_eval = evaluate_agent(
        test_agent,
        real_hospitals,
        episodes=EVAL_EPISODES,
        seed=COMMON_EVAL_SEED,
    )

    controlled_runs[n_plan] = {
        "agent": test_agent,
        "train": test_train,
        "steps": test_steps,
        "eval": test_eval,
    }


# Q-Learning (n=0) defines the common threshold.
qlearning_final100 = (
    controlled_runs[0]["train"]["episode_reward"]
    .tail(100)
    .mean()
)

if qlearning_final100 > 0:
    common_threshold = 0.90 * qlearning_final100
else:
    # If asymptotic reward is nonpositive, use a less-negative 90% target.
    common_threshold = qlearning_final100 + 0.10 * abs(qlearning_final100)

planning_results = []

for n_plan in PLANNING_LEVELS:
    train_df = controlled_runs[n_plan]["train"]
    eval_df = controlled_runs[n_plan]["eval"]

    first_ep = first_episode_to_threshold(
        train_df["episode_reward"],
        common_threshold,
        window=100,
    )

    direct_interactions = (
        first_ep * 12
        if pd.notna(first_ep)
        else np.nan
    )

    planning_results.append({
        "planning_steps": n_plan,
        "average_evaluation_reward": eval_df["episode_reward"].mean(),
        "average_discounted_return": eval_df["discounted_return"].mean(),
        "final_100_training_reward": train_df["episode_reward"].tail(100).mean(),
        "reward_threshold": common_threshold,
        "first_episode_to_threshold": first_ep,
        "direct_interactions_to_threshold": direct_interactions,
        "final_100_direct_td_error": (
            train_df["avg_abs_direct_td_error"].tail(100).mean()
        ),
    })


planning_summary = pd.DataFrame(planning_results)
display(planning_summary)


plt.figure(figsize=(8, 5))
plt.plot(
    planning_summary["planning_steps"],
    planning_summary["average_evaluation_reward"],
    marker="o",
)
plt.xlabel("Planning Steps per Direct Interaction")
plt.ylabel("Average Evaluation Reward")
plt.title("Controlled Effect of Dyna-Q Planning")
plt.show()


plt.figure(figsize=(8, 5))
sample_eff = planning_summary.dropna(
    subset=["direct_interactions_to_threshold"]
)
plt.plot(
    sample_eff["planning_steps"],
    sample_eff["direct_interactions_to_threshold"],
    marker="o",
)
plt.xlabel("Planning Steps per Direct Interaction")
plt.ylabel("Direct Interactions to Reward Threshold")
plt.title("Dyna-Q Sample Efficiency")
plt.show()


# 14. Controlled Hyperparameter Tuning

The purpose of tuning is to identify the configuration that generalizes best to held-out simulated episodes calibrated from CMS data.

## Parameters searched

\[
\alpha \in \{0.05,0.10,0.20\}
\]

\[
\gamma \in \{0.90,0.95,0.99\}
\]

\[
\epsilon_{decay} \in \{0.990,0.995,0.999\}
\]

\[
n_{planning} \in \{0,1,5,10\}
\]

This produces:

\[
3 \times 3 \times 3 \times 4 = 108
\]

candidate configurations.

### Selection principle
The final model is selected using **held-out evaluation results**, not training reward.

Primary ranking:
1. Average evaluation episode reward
2. Average evaluation discounted return
3. Sample efficiency / interactions to threshold
4. Final TD-error stability

Business outcomes (MSPB, readmissions, HVBP efficiency) are reported to explain the winning reward, but are not independently optimized after looking at the results.


In [ ]:

from itertools import product

ALPHA_GRID = [0.05, 0.10, 0.20]
GAMMA_GRID = [0.90, 0.95, 0.99]
EPSILON_DECAY_GRID = [0.990, 0.995, 0.999]
PLANNING_GRID = [0, 1, 5, 10]

COARSE_TRAIN_EPISODES = 800
COARSE_EVAL_EPISODES = 200

# Every configuration faces the same stochastic episode sequence.
COARSE_TRAIN_SEED = 2026
COARSE_EVAL_SEED = 52_026

candidate_grid = list(product(
    ALPHA_GRID,
    GAMMA_GRID,
    EPSILON_DECAY_GRID,
    PLANNING_GRID,
))

print("Number of candidate configurations:", len(candidate_grid))



## 14.1 Reference threshold for sample efficiency

To compare learning speed fairly, a fixed reference model is used:

- \(\alpha=0.10\)
- \(\gamma=0.95\)
- epsilon decay \(=0.995\)
- planning steps \(=0\)

Its final 100-episode average reward defines a common target:

\[
R_{threshold}=0.90 \times R_{\text{reference, final100}}
\]

Every candidate is judged against this same reward threshold.


In [ ]:

def first_episode_to_threshold(rewards, threshold, window=100):
    rolling = rewards.rolling(
        window,
        min_periods=window,
    ).mean()

    hit_positions = np.flatnonzero(
        rolling.to_numpy() >= threshold
    )

    if len(hit_positions) == 0:
        return np.nan

    return int(hit_positions[0] + 1)


reference_agent, reference_train, _ = train_dynaq(
    real_hospitals,
    episodes=COARSE_TRAIN_EPISODES,
    planning_steps=0,
    seed=COARSE_TRAIN_SEED,
    alpha=0.10,
    gamma=0.95,
    epsilon_decay=0.995,
    keep_step_records=False,
)

reference_final100 = (
    reference_train["episode_reward"]
    .tail(100)
    .mean()
)

if reference_final100 > 0:
    COMMON_TUNING_THRESHOLD = 0.90 * reference_final100
else:
    COMMON_TUNING_THRESHOLD = (
        reference_final100
        + 0.10 * abs(reference_final100)
    )

print("Reference final-100 reward:", reference_final100)
print("Common reward threshold:", COMMON_TUNING_THRESHOLD)



## 14.2 Stage 1 — Coarse grid search

All 108 configurations are screened with:
- 800 training episodes
- 200 held-out evaluation episodes
- identical train/evaluation seeds

Step-level histories are disabled during tuning to reduce memory and runtime.


In [ ]:

coarse_results = []

for run_id, (
    alpha,
    gamma,
    epsilon_decay,
    planning_steps,
) in enumerate(candidate_grid, start=1):

    tuned_agent, tuned_train, _ = train_dynaq(
        real_hospitals,
        episodes=COARSE_TRAIN_EPISODES,
        planning_steps=planning_steps,
        seed=COARSE_TRAIN_SEED,
        alpha=alpha,
        gamma=gamma,
        epsilon_decay=epsilon_decay,
        keep_step_records=False,
    )

    tuned_eval = evaluate_agent(
        tuned_agent,
        real_hospitals,
        episodes=COARSE_EVAL_EPISODES,
        seed=COARSE_EVAL_SEED,
    )

    first_ep = first_episode_to_threshold(
        tuned_train["episode_reward"],
        COMMON_TUNING_THRESHOLD,
        window=100,
    )

    interactions = (
        first_ep * 12
        if pd.notna(first_ep)
        else np.nan
    )

    coarse_results.append({
        "run_id": run_id,
        "alpha": alpha,
        "gamma": gamma,
        "epsilon_decay": epsilon_decay,
        "planning_steps": planning_steps,
        "eval_avg_reward": tuned_eval["episode_reward"].mean(),
        "eval_avg_discounted_return": tuned_eval["discounted_return"].mean(),
        "eval_final_mspb": tuned_eval["final_mspb_score"].mean(),
        "eval_final_readmission": tuned_eval["final_readmission_ratio"].mean(),
        "eval_final_hvbp_efficiency": tuned_eval["final_hvbp_efficiency"].mean(),
        "final_100_train_reward": tuned_train["episode_reward"].tail(100).mean(),
        "final_100_direct_td_error": (
            tuned_train["avg_abs_direct_td_error"]
            .tail(100)
            .mean()
        ),
        "first_episode_to_threshold": first_ep,
        "direct_interactions_to_threshold": interactions,
        "epsilon_final": tuned_train["epsilon_end"].iloc[-1],
    })

    if run_id % 12 == 0:
        print(
            f"Completed {run_id}/{len(candidate_grid)} configurations"
        )


coarse_tuning = pd.DataFrame(coarse_results)

coarse_tuning = coarse_tuning.sort_values(
    by=[
        "eval_avg_reward",
        "eval_avg_discounted_return",
        "direct_interactions_to_threshold",
    ],
    ascending=[False, False, True],
    na_position="last",
).reset_index(drop=True)

display(coarse_tuning.head(15))



## 14.3 Inspect the best coarse configurations

The table below makes it easy to see whether the best configurations consistently prefer:
- smaller/larger learning rates,
- shorter/longer discount horizons,
- slower/faster exploration decay,
- Q-Learning (\(n=0\)) or Dyna-Q planning.


In [ ]:

TOP_N = 10
top_coarse = coarse_tuning.head(TOP_N).copy()

display(
    top_coarse[
        [
            "alpha",
            "gamma",
            "epsilon_decay",
            "planning_steps",
            "eval_avg_reward",
            "eval_avg_discounted_return",
            "direct_interactions_to_threshold",
            "final_100_direct_td_error",
            "eval_final_mspb",
            "eval_final_readmission",
            "eval_final_hvbp_efficiency",
        ]
    ]
)

print("\nParameter frequency among top configurations:")

for parameter in [
    "alpha",
    "gamma",
    "epsilon_decay",
    "planning_steps",
]:
    print(f"\n{parameter}")
    print(top_coarse[parameter].value_counts())



## 14.4 Stage 2 — Robust confirmation of the top candidates

A coarse search can identify lucky configurations.  
Therefore, the strongest candidates are rerun using:

- 2,000 training episodes
- 500 evaluation episodes
- three common random seeds

The same three seeds are used for every candidate.

The final model is selected by the **mean held-out evaluation reward across seeds**.


In [ ]:

CONFIRM_TOP_N = 8
CONFIRM_TRAIN_EPISODES = 2000
CONFIRM_EVAL_EPISODES = 500
CONFIRM_SEEDS = [6101, 6102, 6103]

confirm_candidates = (
    coarse_tuning
    .head(CONFIRM_TOP_N)[
        [
            "alpha",
            "gamma",
            "epsilon_decay",
            "planning_steps",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

confirmation_rows = []

for candidate_id, row in confirm_candidates.iterrows():

    for seed in CONFIRM_SEEDS:
        alpha = float(row["alpha"])
        gamma = float(row["gamma"])
        epsilon_decay = float(row["epsilon_decay"])
        planning_steps = int(row["planning_steps"])

        final_agent, final_train, _ = train_dynaq(
            real_hospitals,
            episodes=CONFIRM_TRAIN_EPISODES,
            planning_steps=planning_steps,
            seed=seed,
            alpha=alpha,
            gamma=gamma,
            epsilon_decay=epsilon_decay,
            keep_step_records=False,
        )

        # Same evaluation seed transformation for all candidates.
        final_eval = evaluate_agent(
            final_agent,
            real_hospitals,
            episodes=CONFIRM_EVAL_EPISODES,
            seed=90_000 + seed,
        )

        first_ep = first_episode_to_threshold(
            final_train["episode_reward"],
            COMMON_TUNING_THRESHOLD,
            window=100,
        )

        confirmation_rows.append({
            "candidate_id": candidate_id + 1,
            "seed": seed,
            "alpha": alpha,
            "gamma": gamma,
            "epsilon_decay": epsilon_decay,
            "planning_steps": planning_steps,
            "eval_avg_reward": final_eval["episode_reward"].mean(),
            "eval_avg_discounted_return": final_eval["discounted_return"].mean(),
            "eval_final_mspb": final_eval["final_mspb_score"].mean(),
            "eval_final_readmission": final_eval["final_readmission_ratio"].mean(),
            "eval_final_hvbp_efficiency": final_eval["final_hvbp_efficiency"].mean(),
            "final_100_train_reward": final_train["episode_reward"].tail(100).mean(),
            "final_100_direct_td_error": (
                final_train["avg_abs_direct_td_error"]
                .tail(100)
                .mean()
            ),
            "first_episode_to_threshold": first_ep,
            "direct_interactions_to_threshold": (
                first_ep * 12
                if pd.notna(first_ep)
                else np.nan
            ),
        })


confirmation_runs = pd.DataFrame(confirmation_rows)

confirmation_summary = (
    confirmation_runs
    .groupby(
        [
            "candidate_id",
            "alpha",
            "gamma",
            "epsilon_decay",
            "planning_steps",
        ],
        as_index=False,
    )
    .agg(
        mean_eval_reward=("eval_avg_reward", "mean"),
        sd_eval_reward=("eval_avg_reward", "std"),
        mean_discounted_return=("eval_avg_discounted_return", "mean"),
        mean_final_mspb=("eval_final_mspb", "mean"),
        mean_final_readmission=("eval_final_readmission", "mean"),
        mean_final_hvbp_efficiency=("eval_final_hvbp_efficiency", "mean"),
        mean_final_100_train_reward=("final_100_train_reward", "mean"),
        mean_final_100_td_error=("final_100_direct_td_error", "mean"),
        mean_interactions_to_threshold=("direct_interactions_to_threshold", "mean"),
    )
)

confirmation_summary = confirmation_summary.sort_values(
    by=[
        "mean_eval_reward",
        "mean_discounted_return",
        "mean_interactions_to_threshold",
    ],
    ascending=[False, False, True],
    na_position="last",
).reset_index(drop=True)

display(confirmation_summary)



## 14.5 Select the final tuned configuration

The first row of the confirmation table is the selected configuration.

The selection is based on performance across multiple seeds, not a single lucky training trajectory.


In [ ]:

best = confirmation_summary.iloc[0]

BEST_ALPHA = float(best["alpha"])
BEST_GAMMA = float(best["gamma"])
BEST_EPSILON_DECAY = float(best["epsilon_decay"])
BEST_PLANNING_STEPS = int(best["planning_steps"])

print("SELECTED HYPERPARAMETERS")
print("------------------------")
print("alpha:", BEST_ALPHA)
print("gamma:", BEST_GAMMA)
print("epsilon decay:", BEST_EPSILON_DECAY)
print("planning steps:", BEST_PLANNING_STEPS)
print("mean evaluation reward:", best["mean_eval_reward"])
print("mean discounted return:", best["mean_discounted_return"])
print(
    "mean interactions to threshold:",
    best["mean_interactions_to_threshold"]
)



## 14.6 Train the final tuned model

After model selection, one final model is trained with the selected hyperparameters using 3,000 episodes.

The selected model provides the learning curves, Q-table, and policy for the decision analysis.


In [ ]:

FINAL_TRAIN_SEED = 77_777
FINAL_EVAL_SEED = 88_888

tuned_agent, tuned_training, tuned_steps = train_dynaq(
    real_hospitals,
    episodes=3000,
    planning_steps=BEST_PLANNING_STEPS,
    seed=FINAL_TRAIN_SEED,
    alpha=BEST_ALPHA,
    gamma=BEST_GAMMA,
    epsilon_decay=BEST_EPSILON_DECAY,
    keep_step_records=True,
)

tuned_evaluation = evaluate_agent(
    tuned_agent,
    real_hospitals,
    episodes=500,
    seed=FINAL_EVAL_SEED,
)

tuned_baseline = evaluate_equal_baseline(
    real_hospitals,
    gamma=BEST_GAMMA,
    episodes=500,
    seed=FINAL_EVAL_SEED,
)

final_tuned_comparison = pd.DataFrame({
    "Metric": [
        "Average episode reward",
        "Average discounted return",
        "Final MSPB score",
        "Final excess readmission ratio",
        "Final normalized HVBP MSPB measure score",
    ],
    "Tuned RL": [
        tuned_evaluation["episode_reward"].mean(),
        tuned_evaluation["discounted_return"].mean(),
        tuned_evaluation["final_mspb_score"].mean(),
        tuned_evaluation["final_readmission_ratio"].mean(),
        tuned_evaluation["final_hvbp_efficiency"].mean(),
    ],
    "Equal Allocation": [
        tuned_baseline["episode_reward"].mean(),
        tuned_baseline["discounted_return"].mean(),
        tuned_baseline["final_mspb_score"].mean(),
        tuned_baseline["final_readmission_ratio"].mean(),
        tuned_baseline["final_hvbp_efficiency"].mean(),
    ],
})

final_tuned_comparison["Difference"] = (
    final_tuned_comparison["Tuned RL"]
    - final_tuned_comparison["Equal Allocation"]
)

display(final_tuned_comparison)



## 14.7 Tuning visualizations


In [ ]:

# Best coarse configurations
top_plot = coarse_tuning.head(15).copy()

plt.figure(figsize=(10, 5))
plt.bar(
    range(len(top_plot)),
    top_plot["eval_avg_reward"],
)
plt.xticks(
    range(len(top_plot)),
    [
        f'a={r.alpha}, g={r.gamma}, e={r.epsilon_decay}, n={int(r.planning_steps)}'
        for _, r in top_plot.iterrows()
    ],
    rotation=75,
    ha="right",
)
plt.ylabel("Average Evaluation Reward")
plt.title("Top Coarse Hyperparameter Configurations")
plt.tight_layout()
plt.show()


# Robust confirmation performance
plt.figure(figsize=(9, 5))
plt.errorbar(
    range(len(confirmation_summary)),
    confirmation_summary["mean_eval_reward"],
    yerr=confirmation_summary["sd_eval_reward"],
    marker="o",
    capsize=4,
)
plt.xticks(
    range(len(confirmation_summary)),
    [
        f'a={r.alpha}, g={r.gamma}, e={r.epsilon_decay}, n={int(r.planning_steps)}'
        for _, r in confirmation_summary.iterrows()
    ],
    rotation=75,
    ha="right",
)
plt.ylabel("Mean Evaluation Reward")
plt.title("Confirmed Hyperparameter Performance Across Seeds")
plt.tight_layout()
plt.show()


# Final tuned learning curve
tuned_roll = tuned_training["episode_reward"].rolling(
    100,
    min_periods=20,
).mean()

plt.figure(figsize=(10, 5))
plt.plot(
    tuned_training["episode"],
    tuned_training["episode_reward"],
    alpha=0.20,
)
plt.plot(
    tuned_training["episode"],
    tuned_roll,
    linewidth=2,
)
plt.xlabel("Episode")
plt.ylabel("Episode Reward")
plt.title("Final Tuned RL Learning Curve")
plt.show()


## 15. Export Decision Analysis Results

In [ ]:

OUTPUT = Path("vbhc_policy_results")
OUTPUT.mkdir(exist_ok=True)

real_hospitals.to_csv(
    OUTPUT / "cms_merged_hospital_data.csv",
    index=False,
)

coarse_tuning.to_csv(
    OUTPUT / "coarse_hyperparameter_search_108.csv",
    index=False,
)

confirmation_runs.to_csv(
    OUTPUT / "confirmation_runs_all_seeds.csv",
    index=False,
)

confirmation_summary.to_csv(
    OUTPUT / "confirmation_summary.csv",
    index=False,
)

tuned_training.to_csv(
    OUTPUT / "final_tuned_training_episode_metrics.csv",
    index=False,
)

tuned_steps.to_csv(
    OUTPUT / "final_tuned_training_step_metrics.csv",
    index=False,
)

tuned_evaluation.to_csv(
    OUTPUT / "final_tuned_evaluation.csv",
    index=False,
)

tuned_baseline.to_csv(
    OUTPUT / "final_equal_allocation_baseline.csv",
    index=False,
)

final_tuned_comparison.to_csv(
    OUTPUT / "final_tuned_vs_baseline.csv",
    index=False,
)

pd.DataFrame(tuned_agent.q).to_csv(
    OUTPUT / "final_tuned_q_table.csv",
    index=False,
)

selected_hyperparameters = pd.DataFrame([{
    "alpha": BEST_ALPHA,
    "gamma": BEST_GAMMA,
    "epsilon_decay": BEST_EPSILON_DECAY,
    "planning_steps": BEST_PLANNING_STEPS,
}])

selected_hyperparameters.to_csv(
    OUTPUT / "selected_hyperparameters.csv",
    index=False,
)

print("Decision analysis results saved to:", OUTPUT.resolve())


## 15. Decision Interpretation and Validation Boundaries

### Primary RL evidence
- Higher average episode reward is better.
- Higher discounted return is better.
- Lower direct interactions to the common threshold indicate greater sample efficiency.
- Stabilizing TD error indicates more stable value estimates.

### Business-performance direction
- **Lower MSPB** is better.
- **Lower excess readmission ratio** is better.
- **Higher HVBP efficiency score** is better.

### How to interpret planning

If a Dyna-Q planning condition:
1. achieves equal or higher evaluation reward than \(n=0\), **and**
2. reaches the common reward threshold with fewer direct interactions,

then the results support the practical advantage of planning.

If \(n=0\) remains superior after the controlled experiment, the supported conclusion is that Dyna-Q planning did not improve performance in this CMS-calibrated environment.

### Supported use
The model is a real-data-calibrated RL proof of concept using publicly reported CMS hospital performance measures.

### What it cannot claim
It cannot claim that:
- CMS contains internal hospital resource-allocation decisions,
- the simulated actions caused real patient outcomes,
- the learned policy should be deployed clinically without prospective validation.

### Hyperparameter-tuning interpretation

The selected configuration should be reported as the configuration with the strongest mean held-out evaluation reward across the confirmation seeds.

If the selected planning-step value is \(0\), the empirical conclusion is that **direct Q-Learning outperformed Dyna-Q planning after tuning**.

If the selected planning-step value is greater than \(0\), then Dyna-Q planning demonstrated practical benefit after hyperparameter optimization.
